In [1]:
## NB09: Isolate Prediction — Apply Trained Regression Models to All ENIGMA Proteins

# Apply the 11 trained CatBoostRegressor models (Zn/Cu/Cd/Co/Ni/Cr/Hg/Mn/Fe/Se/Al)
# to all 215,051 proteins from 60 ENIGMA organisms. The core scientific value is
# prediction for organisms that were NEVER experimentally tested for a given metal —
# up to 58 untested organisms for Cd, 51 for Hg, 53 for Se.
#
# Output: ranked candidate lists per metal per organism for experimental follow-up.

import sys, json, logging
from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from scipy.stats import spearmanr

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_labeled_pd, load_best_combination

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
MODEL_DIR = DATA_DIR / 'models'
OUTPUT_DIR = DATA_DIR / 'predictions'
OUTPUT_DIR.mkdir(exist_ok=True)

FITNESS_THRESHOLD = -2.0  # binary-positive cutoff used during training

METAL_STRESSORS = ['Zn', 'Cu', 'Cd', 'Co', 'Ni', 'Cr', 'Hg', 'Mn', 'Fe', 'Se', 'Al']

# Load labeled_pd (index = "{orgId}|{locusId}", cols include organism + _fit cols)
labeled_pd = load_labeled_pd(DATA_DIR)
log.info(f"labeled_pd: {labeled_pd.shape} ({labeled_pd['organism'].nunique()} organisms)")

# Load pre-computed features (same combination used for training)
best_combo = load_best_combination(DATA_DIR)
X_list = []
for name in best_combo:
    df = pd.read_parquet(DATA_DIR / f"features_{name}.parquet").drop(columns=['organism'], errors='ignore')
    X_list.append(df)
X_full = pd.concat(X_list, axis=1)
log.info(f"Feature matrix: {X_full.shape} (combination: {'+'.join(best_combo)})")

# Load regression metrics for reference
reg_metrics = pd.read_csv(DATA_DIR / 'regression_model_metrics.csv', index_col=0)
log.info(f"Regression metrics loaded for: {reg_metrics.index.tolist()}")

2026-06-29 04:07:25,277 INFO labeled_pd: (215051, 96) (60 organisms)


2026-06-29 04:07:25,578 INFO Feature matrix: (215051, 420) (combination: aa+kmer2)


2026-06-29 04:07:25,580 INFO Regression metrics loaded for: ['Zn', 'Cu', 'Cd', 'Co', 'Ni', 'Cr', 'Hg', 'Mn', 'Fe', 'Se', 'Al']


In [2]:
## Apply Regression Models to All 215,051 Proteins

# For each metal, load the trained CatBoostRegressor and predict fitness for
# every protein in labeled_pd. Proteins with NaN {metal}_fit had no experiment —
# these are the novel predictions. Proteins with real fitness values serve as
# a held-out validation check.

pred_cols = {}

for metal in METAL_STRESSORS:
    model_path = MODEL_DIR / f"stressor_{metal}_regression.cbm"
    if not model_path.exists():
        log.warning(f"{metal}: regression model not found; skipping.")
        continue

    model = CatBoostRegressor()
    model.load_model(str(model_path))

    y_pred = model.predict(X_full)  # shape: (215051,)
    pred_cols[f"{metal}_pred"] = y_pred

    # Quick validation: Spearman ρ on proteins with actual fitness data
    fit_col = f"{metal}_fit"
    if fit_col in labeled_pd.columns:
        mask = labeled_pd[fit_col].notna()
        if mask.sum() > 100:
            rho, _ = spearmanr(labeled_pd.loc[mask, fit_col].values, y_pred[mask])
            n_novel = (~mask).sum()
            n_tested_orgs = labeled_pd[mask]['organism'].nunique()
            n_untested_orgs = labeled_pd[~mask]['organism'].nunique()
            log.info(f"{metal}: ρ={rho:.3f} on {mask.sum():,} tested proteins; "
                     f"novel predictions for {n_novel:,} proteins in "
                     f"{n_untested_orgs} untested organisms")

predictions_df = pd.DataFrame(pred_cols, index=labeled_pd.index)
predictions_df.insert(0, 'organism', labeled_pd['organism'])

log.info(f"Prediction matrix: {predictions_df.shape}")
predictions_df.head(3)

2026-06-29 04:07:25,928 INFO Zn: ρ=0.253 on 142,269 tested proteins; novel predictions for 72,782 proteins in 20 untested organisms


2026-06-29 04:07:26,300 INFO Cu: ρ=0.267 on 148,953 tested proteins; novel predictions for 66,098 proteins in 19 untested organisms


2026-06-29 04:07:26,651 INFO Cd: ρ=0.530 on 6,184 tested proteins; novel predictions for 208,867 proteins in 58 untested organisms


2026-06-29 04:07:26,929 INFO Co: ρ=0.279 on 150,081 tested proteins; novel predictions for 64,970 proteins in 19 untested organisms


2026-06-29 04:07:27,252 INFO Ni: ρ=0.271 on 152,937 tested proteins; novel predictions for 62,114 proteins in 18 untested organisms


...

2026-06-29 04:07:28,397 INFO Fe: ρ=0.242 on 115,488 tested proteins; novel predictions for 99,563 proteins in 28 untested organisms


2026-06-29 04:07:28,662 INFO Se: ρ=0.411 on 19,702 tested proteins; novel predictions for 195,349 proteins in 53 untested organisms


2026-06-29 04:07:28,944 INFO Al: ρ=0.215 on 142,561 tested proteins; novel predictions for 72,490 proteins in 21 untested organisms


2026-06-29 04:07:28,947 INFO Prediction matrix: (215051, 12)


,organism,Zn_pred,Cu_pred,Cd_pred,Co_pred,Ni_pred,Cr_pred,Hg_pred,Mn_pred,Fe_pred,Se_pred,Al_pred
protein_id,,,,,,,,,,,,
acidovorax_3H11|Ac3H11_1,acidovorax_3H11,-0.487421,-0.438804,-0.136754,-0.546629,-0.477920,-0.336190,-0.177764,-0.250828,-0.286039,-0.268371,-0.403317
acidovorax_3H11|Ac3H11_10,acidovorax_3H11,-0.463742,-0.577997,-0.243491,-0.625703,-0.498097,-0.355303,-0.236420,-0.216246,-0.381851,-0.215643,-0.357785
acidovorax_3H11|Ac3H11_100,acidovorax_3H11,-0.463868,-0.523648,-0.265937,-0.647420,-0.600779,-0.301109,-0.408008,-0.467171,-0.477272,-0.209514,-0.431248


In [3]:
## Per-Organism Candidate Ranking

# For each metal, rank proteins within each organism by predicted fitness.
# Most negative predicted fitness = highest priority experimental candidate.
# Label each prediction as:
#   'validated'  — protein was tested; actual fitness < -2 (true positive confirmed)
#   'tested'     — protein was tested; actual fitness >= -2 or NaN
#   'novel'      — protein was NEVER tested for this metal (the key prediction)
#
# BUG NOTE: rename pred_col → 'pred_fitness' BEFORE concat, not after,
# to avoid the last-iteration variable clobbering earlier metals' column names.

TOP_K = 20  # candidates per organism per metal

pred_metals = [c.replace('_pred', '') for c in predictions_df.columns if c.endswith('_pred')]
rows = []

for metal in pred_metals:
    pred_col = f"{metal}_pred"
    fit_col = f"{metal}_fit"

    for org, grp in predictions_df.groupby('organism'):
        # Select and rename to unified column BEFORE building the row
        ranked = grp[[pred_col]].rename(columns={pred_col: 'pred_fitness'}).copy()
        ranked['organism'] = org
        ranked['metal'] = metal

        # Label: novel / tested / validated
        if fit_col in labeled_pd.columns:
            fit_vals = labeled_pd.loc[grp.index, fit_col]
            ranked['actual_fitness'] = fit_vals.values
            ranked['status'] = np.where(
                fit_vals.isna(), 'novel',
                np.where(fit_vals < FITNESS_THRESHOLD, 'validated', 'tested')
            )
        else:
            ranked['actual_fitness'] = np.nan
            ranked['status'] = 'novel'

        ranked = ranked.sort_values('pred_fitness').head(TOP_K)
        ranked['rank'] = range(1, len(ranked) + 1)
        ranked['locusId'] = ranked.index.str.split('|').str[1]
        rows.append(ranked.reset_index(names='protein_id'))

candidates_df = pd.concat(rows, ignore_index=True)
# All rows now have a single clean 'pred_fitness' column
assert 'pred_fitness' in candidates_df.columns
assert candidates_df['pred_fitness'].notna().all(), "pred_fitness should be non-null for all candidates"

# Summary: how many novel top-20 candidates per metal?
print("Top-20 candidates per metal per organism — status breakdown:")
summary = candidates_df.groupby(['metal', 'status']).size().unstack(fill_value=0)
print(summary.to_string())
print()

# Novel candidates only (the primary scientific output)
novel = candidates_df[candidates_df['status'] == 'novel']
print(f"Novel candidates (untested organisms): {len(novel):,} across {novel['metal'].nunique()} metals")
print(f"  Predicted fitness range: {novel['pred_fitness'].min():.3f} to {novel['pred_fitness'].max():.3f}")
print(f"  Below -2.0 threshold (predicted positive): {(novel['pred_fitness'] < -2.0).sum():,} ({(novel['pred_fitness'] < -2.0).mean():.1%})")
novel_by_metal = novel.groupby('metal')['organism'].nunique()
print(novel_by_metal.to_string())

Top-20 candidates per metal per organism — status breakdown:
status  novel  tested  validated
metal                           
Al        420     336        444
Cd       1160      19         21
Co        380     305        515
Cr        980      89        131
Cu        380     283        537
Fe        560     366        274
Hg       1020      42        138
Mn       1120      20         60
Ni        360     273        567
Se       1060      38        102
Zn        400     352        448

Novel candidates (untested organisms): 7,840 across 11 metals
  Predicted fitness range: -3.900 to -0.600
  Below -2.0 threshold (predicted positive): 106 (1.4%)
metal
Al    21
Cd    58
Co    19
Cr    49
Cu    19
Fe    28
Hg    51
Mn    56
Ni    18
Se    53
Zn    20


In [4]:
## Multi-Metal Overlap: Broad-Spectrum Metal Resistance Candidates

# Proteins predicted to be fitness-deficient under MULTIPLE metals are the most
# compelling candidates — they likely encode core metal homeostasis machinery
# (efflux pumps, metal-binding chaperones) rather than metal-specific resistance.
#
# Multi-metal score = number of metals for which this protein ranks in the
# top-K (TOP_K=20) for its organism.

# Build per-protein multi-metal rank matrix
pred_metal_cols = [f"{m}_pred" for m in pred_metals if f"{m}_pred" in predictions_df.columns]

# For each protein, compute: how many metals put it in top-K per organism?
TOP_K_MULTI = 50  # wider window for multi-metal analysis
multi_rows = []

for org, grp in predictions_df.groupby('organism'):
    n = len(grp)
    top_k = min(TOP_K_MULTI, n)

    metal_flags = {}
    for col in pred_metal_cols:
        metal = col.replace('_pred', '')
        # rank within organism (lower pred_fitness = higher rank)
        ranks = grp[col].rank(method='min', ascending=True)
        metal_flags[metal] = (ranks <= top_k).astype(int)

    flag_df = pd.DataFrame(metal_flags, index=grp.index)
    flag_df['multi_metal_score'] = flag_df.sum(axis=1)
    flag_df['organism'] = org
    multi_rows.append(flag_df)

multi_df = pd.concat(multi_rows)
multi_df['locusId'] = multi_df.index.str.split('|').str[1]

# Annotate with novel/tested status (use any available fit column to check)
# A protein is 'novel across ALL metals' if it has no fit data for any metal
fit_cols_present = [f"{m}_fit" for m in pred_metals if f"{m}_fit" in labeled_pd.columns]
any_tested = labeled_pd[fit_cols_present].notna().any(axis=1)
multi_df['any_tested'] = any_tested.reindex(multi_df.index).values

# Top multi-metal candidates
top_multi = (multi_df[multi_df['multi_metal_score'] >= 3]
             .sort_values('multi_metal_score', ascending=False))

print(f"Proteins in top-{TOP_K_MULTI} for ≥3 metals (per organism): {len(top_multi):,}")
print(f"  Of which completely untested (no fit data for any metal): "
      f"{(~top_multi['any_tested']).sum():,}")
print()

# Show distribution of multi-metal scores
score_dist = multi_df['multi_metal_score'].value_counts().sort_index()
print("Multi-metal score distribution (0 = never top-50, 11 = top-50 for all metals):")
for score, count in score_dist.items():
    bar = '█' * min(40, count // 200)
    print(f"  {score:2d} metals: {count:6,}  {bar}")
print()

# Top-20 multi-metal proteins across all organisms
print("Top 20 proteins by multi-metal score (any tested status):")
cols_to_show = ['organism', 'locusId', 'multi_metal_score'] + pred_metals
top20_multi = top_multi.head(20)[cols_to_show] if all(m in top_multi.columns for m in pred_metals) else top_multi.head(20)
print(top20_multi.to_string())

Proteins in top-50 for ≥3 metals (per organism): 3,784
  Of which completely untested (no fit data for any metal): 887

Multi-metal score distribution (0 = never top-50, 11 = top-50 for all metals):
   0 metals: 199,657  ████████████████████████████████████████
   1 metals:  9,592  ████████████████████████████████████████
   2 metals:  2,018  ██████████
   3 metals:  1,008  █████
   4 metals:    703  ███
   5 metals:    584  ██
   6 metals:    544  ██
   7 metals:    448  ██
   8 metals:    298  █
   9 metals:    159  
  10 metals:     39  
  11 metals:      1  

Top 20 proteins by multi-metal score (any tested status):
                                              organism          locusId  multi_metal_score  Zn  Cu  Cd  Co  Ni  Cr  Hg  Mn  Fe  Se  Al
protein_id                                                                                                                            
acidovorax_3H11|Ac3H11_638             acidovorax_3H11       Ac3H11_638                 11   1   1   1

In [5]:
## Validation: Predicted vs. Actual Fitness on Tested Proteins (TRAIN/TEST Labeled)

# Per-organism Spearman ρ between predicted and actual fitness.
# Critical distinction: organisms in the TRAIN set give in-sample ρ (inflated),
# while TEST organisms give genuinely held-out estimates of generalization.
# Load the test group from each metal's reg_predictions.parquet to label correctly.

from scipy.stats import spearmanr

# Build lookup: metal → set of test organism IDs
test_orgs_by_metal = {}
for metal in pred_metals:
    reg_pred_path = MODEL_DIR / f"stressor_{metal}_reg_predictions.parquet"
    if reg_pred_path.exists():
        rp = pd.read_parquet(reg_pred_path)
        test_orgs_by_metal[metal] = set(rp['group'].unique())
    else:
        test_orgs_by_metal[metal] = set()

print("Per-organism Spearman ρ (predicted vs. actual fitness):\n")
print(f"{'Metal':<6} {'Split':<6} {'Organism':<35} {'n_tested':>8} {'ρ':>7} {'p':>10}")
print("-" * 78)

for metal in pred_metals:
    fit_col = f"{metal}_fit"
    pred_col = f"{metal}_pred"
    if fit_col not in labeled_pd.columns:
        continue

    test_orgs = test_orgs_by_metal.get(metal, set())

    for org, grp in labeled_pd.groupby('organism'):
        fit_vals = grp[fit_col].dropna()
        if len(fit_vals) < 30:
            continue
        pred_vals = predictions_df.loc[fit_vals.index, pred_col]
        rho, pval = spearmanr(fit_vals.values, pred_vals.values)
        sig = '*' if pval < 0.05 else ''
        split = 'TEST' if org in test_orgs else 'TRAIN'
        print(f"{metal:<6} {split:<6} {org:<35} {len(fit_vals):>8,} {rho:>7.3f} {pval:>10.3e} {sig}")

print()
print("Note: TRAIN = organism was in the training set (ρ is in-sample, inflated).")
print("      TEST  = organism was held out (ρ is a genuine generalization estimate).")
print("Only TEST ρ values should be used to assess predictive validity.")

Per-organism Spearman ρ (predicted vs. actual fitness):

Metal  Split  Organism                            n_tested       ρ          p
------------------------------------------------------------------------------


Zn     TRAIN  BFirm                                  5,387   0.215  1.595e-57 *
Zn     TRAIN  Bifido                                 1,599   0.153  8.153e-10 *
Zn     TRAIN  Btheta                                 4,036   0.388 1.887e-145 *
Zn     TRAIN  Burk376                                5,081   0.099  1.227e-12 *
Zn     TEST   Caulo                                  3,289  -0.013  4.487e-01 
Zn     TRAIN  Cup4G11                                6,336   0.288 1.436e-121 *
Zn     TRAIN  Dino                                   3,181   0.173  6.765e-23 *
Zn     TRAIN  DvH                                    2,716   0.306  5.798e-60 *
Zn     TRAIN  HerbieS                                3,861   0.233  7.741e-49 *
Zn     TRAIN  Kang                                   1,971   0.090  6.390e-05 *


Zn     TRAIN  Keio                                   3,462   0.249  4.095e-50 *
Zn     TRAIN  Korea                                  3,336   0.334  1.175e-87 *
Zn     TEST   Koxy                                   4,450   0.141  3.847e-21 *
Zn     TRAIN  MR1                                    3,635   0.093  1.997e-08 *
Zn     TRAIN  Magneto                                3,145   0.112  2.916e-10 *
Zn     TEST   Marino                                 3,630   0.211  1.097e-37 *
Zn     TEST   Methanococcus_JJ                       1,350   0.386  3.005e-49 *
Zn     TRAIN  Methanococcus_S2                       1,223   0.474  2.094e-69 *
Zn     TRAIN  Miya                                   2,506   0.299  4.408e-53 *
Zn     TEST   MycoTube                               2,856   0.085  6.017e-06 *
Zn     TRAIN  PS                                     2,537  -0.030  1.337e-01 
Zn     TRAIN  PV4                                    2,982   0.137  6.157e-14 *
Zn     TRAIN  Pedo557                    

Cu     TRAIN  BFirm                                  5,387   0.188  7.406e-44 *
Cu     TRAIN  Btheta                                 4,036   0.387 1.969e-144 *
Cu     TRAIN  Burk376                                5,081   0.134  7.929e-22 *
Cu     TRAIN  Caulo                                  3,289   0.077  9.728e-06 *
Cu     TEST   Cup4G11                                6,336   0.227  1.357e-74 *
Cu     TRAIN  Dino                                   3,181   0.157  6.205e-19 *
Cu     TRAIN  DvH                                    2,716   0.260  2.419e-43 *
Cu     TRAIN  HerbieS                                3,861   0.246  1.834e-54 *
Cu     TEST   Kang                                   1,971   0.175  5.220e-15 *
Cu     TRAIN  Keio                                   3,462   0.287  7.166e-67 *
Cu     TRAIN  Korea                                  3,336   0.348  7.645e-96 *
Cu     TRAIN  Koxy                                   4,450   0.210  9.830e-46 *
Cu     TRAIN  MR1                       

Cd     TRAIN  psRCH2                                 3,306   0.782  0.000e+00 *
Cd     TEST   rhodanobacter_10B01                    2,878   0.199  3.239e-27 *
Co     TRAIN  BFirm                                  5,387   0.245  2.265e-74 *
Co     TRAIN  Btheta                                 4,036   0.381 2.253e-139 *
Co     TRAIN  Burk376                                5,081   0.162  3.176e-31 *
Co     TRAIN  Caulo                                  3,289   0.172  3.403e-23 *
Co     TEST   Cup4G11                                6,336   0.271 1.094e-106 *
Co     TRAIN  Dino                                   3,181   0.192  9.078e-28 *
Co     TRAIN  DvH                                    2,716   0.210  1.654e-28 *


...

Hg     TRAIN  Btheta                                 4,036   0.355 1.827e-120 *
Hg     TEST   DvH                                    2,716   0.051  7.521e-03 *
Hg     TRAIN  Koxy                                   4,450   0.273  1.139e-76 *
Hg     TRAIN  Marino                                 3,630   0.272  1.311e-62 *
Hg     TRAIN  Methanococcus_S2                       1,223   0.530  1.981e-89 *
Hg     TRAIN  Miya                                   2,506   0.332  2.200e-65 *
Hg     TRAIN  Phaeo                                  3,084   0.259  2.411e-48 *
Hg     TEST   Putida                                 4,740   0.067  4.474e-06 *
Hg     TRAIN  rhodanobacter_10B01                    2,878   0.337  2.619e-77 *


Mn     TRAIN  Btheta                                 4,036   0.544 2.251e-309 *
Mn     TEST   DvH                                    2,716   0.050  8.979e-03 *
Mn     TRAIN  Putida                                 4,740   0.424 7.851e-206 *
Mn     TRAIN  rhodanobacter_10B01                    2,878   0.570 9.136e-248 *


Fe     TRAIN  BFirm                                  5,387   0.224  3.158e-62 *
Fe     TRAIN  Btheta                                 4,036   0.403 1.419e-157 *
Fe     TRAIN  Burk376                                5,081   0.163  1.440e-31 *
Fe     TRAIN  Caulo                                  3,289   0.085  1.075e-06 *
Fe     TRAIN  Dino                                   3,181   0.228  1.003e-38 *
Fe     TRAIN  DvH                                    2,716   0.248  1.778e-39 *
Fe     TRAIN  HerbieS                                3,861   0.200  5.178e-36 *
Fe     TRAIN  Keio                                   3,462   0.193  1.949e-30 *
Fe     TEST   Korea                                  3,336   0.267  1.267e-55 *
Fe     TEST   Koxy                                   4,450   0.067  8.969e-06 *
Fe     TRAIN  Magneto                                3,145   0.105  3.469e-09 *
Fe     TRAIN  Marino                                 3,630   0.166  8.152e-24 *
Fe     TRAIN  Methanococcus_JJ          

Se     TEST   Btheta                                 4,036   0.212  3.430e-42 *
Se     TEST   DvH                                    2,716   0.095  7.327e-07 *
Se     TRAIN  Koxy                                   4,450   0.350 1.438e-128 *
Se     TRAIN  Methanococcus_S2                       1,223   0.612 9.756e-127 *
Se     TRAIN  Miya                                   2,506   0.424 1.156e-109 *
Se     TRAIN  SynE                                   1,893   0.509 2.804e-125 *
Se     TRAIN  rhodanobacter_10B01                    2,878   0.448 5.648e-142 *


Al     TRAIN  BFirm                                  5,387   0.148  1.292e-27 *
Al     TRAIN  Btheta                                 4,036   0.377 1.028e-136 *
Al     TRAIN  Burk376                                5,081   0.138  5.014e-23 *
Al     TRAIN  Caulo                                  3,289   0.035  4.211e-02 *
Al     TEST   Cup4G11                                6,336   0.152  2.996e-34 *
Al     TRAIN  Dino                                   3,181   0.159  1.893e-19 *
Al     TEST   DvH                                    2,716   0.184  3.465e-22 *
Al     TRAIN  HerbieS                                3,861   0.218  6.532e-43 *
Al     TRAIN  Kang                                   1,971   0.140  4.823e-10 *
Al     TRAIN  Keio                                   3,462   0.257  2.072e-53 *
Al     TRAIN  Korea                                  3,336   0.270  5.615e-57 *
Al     TRAIN  Koxy                                   4,450   0.172  9.583e-31 *
Al     TRAIN  MR1                       

In [6]:
## Save Outputs

# 1. Full prediction matrix (all proteins × all metals)
pred_out = predictions_df.copy()
# Add actual fitness for reference (NaN where untested)
for metal in pred_metals:
    fit_col = f"{metal}_fit"
    if fit_col in labeled_pd.columns:
        pred_out[fit_col] = labeled_pd[fit_col]
pred_out.to_parquet(OUTPUT_DIR / 'all_protein_predictions.parquet')
log.info(f"Saved full prediction matrix: {pred_out.shape} → {OUTPUT_DIR / 'all_protein_predictions.parquet'}")

# 2. Top-K candidates per metal per organism (with novel/tested/validated status)
candidates_df.to_csv(OUTPUT_DIR / 'top_candidates_per_metal_per_org.csv', index=False)
log.info(f"Saved candidate list: {len(candidates_df):,} rows → {OUTPUT_DIR / 'top_candidates_per_metal_per_org.csv'}")

# 3. Multi-metal overlap scores
multi_out = multi_df[['organism', 'locusId', 'multi_metal_score', 'any_tested'] + pred_metals]
multi_out.to_csv(OUTPUT_DIR / 'multi_metal_scores.csv')
log.info(f"Saved multi-metal scores: {len(multi_out):,} rows → {OUTPUT_DIR / 'multi_metal_scores.csv'}")

# 4. Novel-only candidate summary (top-10 per metal per untested organism)
novel_candidates = candidates_df[candidates_df['status'] == 'novel'].copy()
novel_candidates.to_csv(OUTPUT_DIR / 'novel_candidates.csv', index=False)
log.info(f"Saved novel-only candidates: {len(novel_candidates):,} rows → {OUTPUT_DIR / 'novel_candidates.csv'}")

print("\nOutput files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    mb = f.stat().st_size / 1e6
    print(f"  {f.name}: {mb:.1f} MB")

2026-06-29 04:07:36,148 INFO Saved full prediction matrix: (215051, 23) → /home/hmacgregor/BERIL-research-observatory/projects/enigma_stress_phenotype_ml/data/predictions/all_protein_predictions.parquet


2026-06-29 04:07:36,188 INFO Saved candidate list: 13,200 rows → /home/hmacgregor/BERIL-research-observatory/projects/enigma_stress_phenotype_ml/data/predictions/top_candidates_per_metal_per_org.csv


2026-06-29 04:07:36,586 INFO Saved multi-metal scores: 215,051 rows → /home/hmacgregor/BERIL-research-observatory/projects/enigma_stress_phenotype_ml/data/predictions/multi_metal_scores.csv


2026-06-29 04:07:36,610 INFO Saved novel-only candidates: 7,840 rows → /home/hmacgregor/BERIL-research-observatory/projects/enigma_stress_phenotype_ml/data/predictions/novel_candidates.csv



Output files:
  all_protein_predictions.parquet: 28.7 MB
  multi_metal_scores.csv: 16.1 MB
  novel_candidates.csv: 0.6 MB
  top_candidates_per_metal_per_org.csv: 1.1 MB


In [7]:
## Prediction Summary Table: Model Coverage and Confidence

# Show per-metal model quality and coverage to inform how much to trust
# the predictions for each stressor.

summary_rows = []
for metal in pred_metals:
    fit_col = f"{metal}_fit"
    pred_col = f"{metal}_pred"

    row = {'Metal': metal}

    # From training metrics
    if metal in reg_metrics.index:
        row['Spearman_ρ'] = reg_metrics.loc[metal, 'Spearman_rho']
        row['AUC_ranking'] = reg_metrics.loc[metal, 'AUC_from_ranking']
        row['Train_orgs'] = int(reg_metrics.loc[metal, 'n_orgs'])

    # Coverage
    if fit_col in labeled_pd.columns:
        mask = labeled_pd[fit_col].notna()
        row['Tested_orgs'] = labeled_pd[mask]['organism'].nunique()
        row['Untested_orgs'] = labeled_pd[~mask]['organism'].nunique()
        row['Novel_proteins'] = int((~mask).sum())
    else:
        row['Tested_orgs'] = 0
        row['Untested_orgs'] = labeled_pd['organism'].nunique()
        row['Novel_proteins'] = len(labeled_pd)

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('Metal')
summary = summary.sort_values('Spearman_ρ', ascending=False)

print("Model quality and prediction coverage per metal:\n")
print(summary.to_string(float_format='{:.3f}'.format))
print()
print("Note: Spearman ρ and AUC_ranking measured on held-out test organisms.")
print("Higher AUC_ranking means the model more reliably puts known fitness-deficient")
print("proteins near the top when ranking all proteins in a test organism.")
print()
print("Interpretation guide:")
print("  Hg (AUC=0.774, ρ=0.175): high confidence — strong sequence signal for Hg resistance")
print("  Cd (AUC=0.556, ρ=0.199): lower confidence — only 2 training organisms (anaerobic Cd)")
print("  As/Pb/Ag: not trained — only 1 organism has data; treat any predictions as untested")

Model quality and prediction coverage per metal:

       Spearman_ρ  AUC_ranking  Train_orgs  Tested_orgs  Untested_orgs  Novel_proteins
Metal                                                                                 
Co          0.230        0.679          41           41             19           64970
Cd          0.199        0.556           2            2             58          208867
Cu          0.198        0.655          41           41             19           66098
Zn          0.195        0.688          40           40             20           72782
Hg          0.175        0.774           9            9             51          185788
Ni          0.159        0.663          42           42             18           62114
Se          0.155        0.699           7            7             53          195349
Fe          0.148        0.627          32           32             28           99563
Cr          0.141        0.671          11           11             49          